## Terminology Update

Some historical output logs in this notebook were captured before the current naming migration.
Current runtime terminology in the codebase is:
- `single_csv`
- `multi_csv`
- `ExecutionContext`


In [1]:
# Setup: add project root
import sys
import os
sys.path.insert(0, '..')

from src.orchestrator import Orchestrator
from src.standards import METADATA_STANDARDS
from src.context.context_factory import create_context
BASE = os.path.abspath(os.path.join('..'))

import logging

# Setup logging for Jupyter notebook
# Force reconfigure by removing existing handlers
logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Remove existing handlers to avoid duplicates
for handler in logger.handlers[:]:
    logger.removeHandler(handler)

# Add a fresh StreamHandler that outputs to stdout (visible in notebook)
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(name)s - %(message)s'))
logger.addHandler(handler)

STANDARD_NAME = "croissant_standard_subset"
# STANDARD_NAME = "spatial_ecological"

print("Imports OK")

Python-dotenv could not parse statement starting at line 5


Imports OK


In [2]:
from src.orchestrator import run_metadata_extraction
multi_source = {
    'biota': os.path.join(BASE, 'data/biota_dataset/biota.csv'),
    'samples': os.path.join(BASE, 'data/biota_dataset/samples.csv'),
    'species': os.path.join(BASE, 'data/biota_dataset/species.csv'),
}

# multi_source = {
#     'event': os.path.join(BASE, 'data/protoDT/annual_budburst_per_tree.csv'),
#     'occurrence': os.path.join(BASE, 'data/protoDT/budburst_climwin_input.csv'),
#     'temp': os.path.join(BASE, 'data/protoDT/temp_climwin_input.csv'),
# }

# multi_source = {
#     'event': os.path.join(BASE, 'data/cropharvest/cropharvest_cleaned_global.csv')
# }

# multi_source = {
#     'event': os.path.join(BASE, 'data/S2BMS/bms_presence_y-2018-2019_th-200.csv'),
# }

In [3]:
data_context = create_context(
    source=multi_source,
    name='biota_multi'
)
# metadata_standard=METADATA_STANDARDS['spatial_ecological']
metadata_standard = METADATA_STANDARDS[STANDARD_NAME]
orchestrator = Orchestrator(topology_name='single')
plan = orchestrator.generate_plan(
    context=data_context,
    metadata_standard=metadata_standard
)

/tmp/ipykernel_102604/3010854621.py:7: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  orchestrator = Orchestrator(topology_name='single')


2026-06-22 13:58:21,133 - INFO - root - PlanExecutor initialized with topology: single
2026-06-22 13:58:21,134 - INFO - root -   Players per step: 1
2026-06-22 13:58:21,134 - INFO - root -   Debate rounds: 0
2026-06-22 13:58:21,134 - INFO - root -   Player pool: ['dataset_schema_preview', 'data_analyst', 'schema_expert', 'metadata_specialist', 'relationship_analyst', 'spatial_temporal_specialist', 'critic']
2026-06-22 13:58:21,135 - INFO - root - Orchestrator initialized with topology: single
2026-06-22 13:58:21,135 - INFO - root - [ui] planning
2026-06-22 13:58:21,135 - INFO - root - ============================================================
2026-06-22 13:58:21,135 - INFO - root - GENERATING PLAN
2026-06-22 13:58:21,135 - INFO - root - Context: biota_multi
2026-06-22 13:58:21,135 - INFO - root - Context type: multi_csv
2026-06-22 13:58:21,136 - INFO - root - Classified planning type: multi_csv
2026-06-22 13:58:21,136 - INFO - root - Resources: ['biota', 'samples', 'species']
2026-06

/home/com3dian/Github/metadata_agent/notebooks/../src/players/player.py:822: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return Player(


2026-06-22 13:58:24,805 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-06-22 13:58:24,831 - WARNING - root - Plan validation warning: Step 3 ('Generate the final Croissant metadata JSON object using schema, relationships, and spatial-temporal artifacts.') has unmet dependencies. Missing artifacts: {'metadata_standard'}
2026-06-22 13:58:24,832 - INFO - root - Plan generated successfully!
2026-06-22 13:58:24,832 - INFO - root - Number of steps: 4
2026-06-22 13:58:24,832 - INFO - root -   Step 1: Get column structure and first row samples for all resources to establish schema baseline. (player: dataset_schema_preview)
2026-06-22 13:58:24,833 - INFO - root -   Step 2: Discover and validate relationships between biota, samples, and species resources. (player: relationship_analyst) (resources: ['biota', 'samples', 'species'])
2026-06-22 13:58:24,833 - INFO - root -   Step 3: Detect spatial and temporal columns, then extract concrete 

In [4]:
# Validate plan dataflow
from src.orchestrator.utils import validate_plan_dataflow

if plan:
    # Convert to dict list for validation
    plan_dicts = plan.to_dict_list()
    
    is_valid, message = validate_plan_dataflow(plan_dicts)
    
    if is_valid:
        print(f"✅ {message}")
    else:
        print(f"❌ {message}")
else:
    print("No plan to validate.")

✅ Plan dataflow is valid.


In [5]:
plan.pretty_print()

Plan Steps:
Step 0: Get column structure and first row samples for all resources to establish schema baseline.
  Rationale: Capture the schema and representative data for all three resources (biota, samples, species) in a single call to inform downstream relationship and metadata generation.
  Required Artifacts: {}
  Produced Artifacts: ['schema_preview']
Step 1: Discover and validate relationships between biota, samples, and species resources.
  Rationale: Identify primary keys, foreign keys, and join conditions (biota->samples, biota->species) required for constructing recordset linkages in the metadata.
  Required Artifacts: {'schema_preview': 'schema_preview'}
  Produced Artifacts: ['relationships']
Step 2: Detect spatial and temporal columns, then extract concrete spatial extent and temporal range values.
  Rationale: The metadata standard requires spatialCoverage and temporalCoverage. We must detect which columns contain coordinates and dates, then extract the actual min/max val

In [6]:
from src.orchestrator.plan_executor import PlanExecutor
from src.tools.context_tools import register_context

context_key = "ctx_biota_multi"
register_context(context_key, data_context)
executor = PlanExecutor(topology_name="single")
result = executor.execute(
    plan=plan,
    context=data_context,
    context_key=context_key,
    metadata_standard=metadata_standard,
    metadata_standard_name=STANDARD_NAME
)

2026-06-22 13:58:24,857 - INFO - root - PlanExecutor initialized with topology: single
2026-06-22 13:58:24,857 - INFO - root -   Players per step: 1
2026-06-22 13:58:24,858 - INFO - root -   Debate rounds: 0
2026-06-22 13:58:24,858 - INFO - root -   Player pool: ['dataset_schema_preview', 'data_analyst', 'schema_expert', 'metadata_specialist', 'relationship_analyst', 'spatial_temporal_specialist', 'critic']
2026-06-22 13:58:24,858 - INFO - root - Using structured output schema: CroissantStandardSubsetMetadata
2026-06-22 13:58:24,858 - INFO - root - ============================================================
2026-06-22 13:58:24,859 - INFO - root - STARTING PLAN EXECUTION
2026-06-22 13:58:24,859 - INFO - root - Context: biota_multi
2026-06-22 13:58:24,859 - INFO - root - Type: multi_csv
2026-06-22 13:58:24,859 - INFO - root - Resources: ['biota', 'samples', 'species']
2026-06-22 13:58:24,860 - INFO - root - Steps: 4
2026-06-22 13:58:24,860 - INFO - root - ===============================

/home/com3dian/Github/metadata_agent/notebooks/../src/players/player.py:822: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return Player(


2026-06-22 13:58:25,487 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-06-22 13:58:25,497 - INFO - root - Player 'dataset_schema_preview_1': LLM tool loop iteration 1 — model requested 1 tool(s): get_columns_with_first_row
2026-06-22 13:58:31,243 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-06-22 13:58:31,245 - INFO - root - Player 'dataset_schema_preview_1': LLM tool loop finished after 2 iteration(s), 1 tool invocation(s) — model returned final analysis
2026-06-22 13:58:31,245 - INFO - root - Player 'dataset_schema_preview_1': using LLM tool path — 1 tool result(s), 1 trace entry/entries
2026-06-22 13:58:31,245 - INFO - root -   Player 'dataset_schema_preview_1' completed execution
2026-06-22 13:58:31,246 - INFO - root -     Output: ## Schema Baseline Analysis  ### Approach Used `get_columns_with_first_row` with no resource filter to capture the complete column stru

/home/com3dian/Github/metadata_agent/notebooks/../src/players/player.py:822: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return Player(


2026-06-22 13:58:36,893 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-06-22 13:58:36,895 - INFO - root - Player 'relationship_analyst_1': LLM tool loop finished after 1 iteration(s), 0 tool invocation(s) — model returned final analysis
2026-06-22 13:58:36,896 - INFO - root - Player 'relationship_analyst_1': using LLM tool path — 0 tool result(s), 0 trace entry/entries
2026-06-22 13:58:36,896 - INFO - root -   Player 'relationship_analyst_1' completed execution
2026-06-22 13:58:36,897 - INFO - root -     Output: I'll analyze the relationships between biota, samples, and species resources by examining their schemas and data patterns.  
2026-06-22 13:58:36,897 - INFO - root - Single player, skipping debate
2026-06-22 13:58:36,898 - INFO - root - --- STEP 1: SYNTHESIS ---
2026-06-22 13:58:38,656 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-06-22 13:58:38,658 - INFO - roo

In [7]:
from pprint import pprint
pprint(result.final_workspace['metadata_output'])

{'description': 'A dataset containing biological observations (biota) of '
                'species in tidal flats, linked to sampling events (samples) '
                'and species metadata (species).',
 'filesets': [{'excludes': [],
               'includes': ['biota.csv', 'samples.csv', 'species.csv']}],
 'inLanguage': ['en'],
 'keywords': ['biota',
              'tidal flats',
              'species',
              'samples',
              'ecology',
              'Wadden Sea'],
 'name': 'Biota Multi Dataset',
 'recordsets': [{'examples': ['sample_id: 33941, sampling_station_id: 59, '
                              'sampling_type: grid, date: 2008-07-14, '
                              'platform: boat, tidal_basin_name: Marsdiep, '
                              'tidal_flat_name: Vlakte van Kerken, x: '
                              '4.902667, y: 53.089333, median_grain_size: '
                              'null, percentage_mud: null'],
                 'fields': [{'arrayShape': No

In [8]:
result.final_workspace

{'metadata_standard': '{\n    "name": "Name of the dataset.",\n    "description": "Explain what the dataset contains.",\n    "keywords": "List search terms or tags.",\n    "filesets": "One object per file or file collection.",\n    "recordsets": "One object per table or entity.",\n    "inLanguage": "Specify ISO 639 language codes appearing in the data, e.g. [\'en\'], [\'en\', \'fr\'], [\'zh-CN\'].",\n    "spatialCoverage": "Geographic bounding box in WGS84 with numeric fields: min_lat, min_lon, max_lat, max_lon",\n    "temporalCoverage": "Time period covered, from and to date"\n}',
 'schema_preview': '{\n  "biota": {\n    "columns": [\n      "sample_id",\n      "sibes_id",\n      "abundance_m2",\n      "afdm_m2"\n    ],\n    "first_row": {\n      "sample_id": 33941.0,\n      "sibes_id": 16.0,\n      "abundance_m2": 28.8716,\n      "afdm_m2": 0.4331\n    }\n  },\n  "samples": {\n    "columns": [\n      "sample_id",\n      "sampling_station_id",\n      "sampling_type",\n      "date",\n  

In [11]:
executor.list_tools_called()

['get_columns_with_first_row']

In [12]:
executor.find_tool_calls('get_columns_with_first_row')

[{'agent': 'dataset_schema_preview_1',
  'input': {'context_key': 'ctx_biota_multi', 'resource': ''},
  'output': '{\n  "biota": {\n    "columns": [\n      "sample_id",\n      "sibes_id",\n      "abundance_m2",\n      "afdm_m2"\n    ],\n    "first_row": {\n      "sample_id": 33941.0,\n      "sibes_id": 16.0,\n      "abundance_m2": 28.8716,\n      "afdm_m2": 0.4331\n    }\n  },\n  "samples": {\n    "columns": [\n      "sample_id",\n      "sampling_station_id",\n      "sampling_type",\n      "date",\n      "platform",\n      "tidal_basin_name",\n      "tidal_flat_name",\n      "x",\n      "y",\n      "median_grain_size",\n      "percentage_mud"\n    ],\n    "first_row": {\n      "sample_id": 33941,\n      "sampling_station_id": 59,\n      "sampling_type": "grid",\n      "date": "2008-07-14",\n      "platform": "boat",\n      "tidal_basin_name": "Marsdiep",\n      "tidal_flat_name": "Vlakte van Kerken",\n      "x": 4.902667,\n      "y": 53.089333,\n      "median_grain_size": null,\n      